# Liczby pierwsze

In [1]:
import math
import random
import matplotlib.pyplot as plt

## Testowanie pierwszości

Głównym tematem dzisiejszych laboratoriów będą **testy pierwszości** (primality tests), tj. algorytmy sprawdzania, czy zadana liczba jest liczbą pierwszą. Jakkolwiek istnieją asymptotycznie wydajne *deterministyczne* algorytmy rozwiązujące ten problem, w praktyce są one znacznie wolniejsze i bardziej skomplikowane od algorytmów *probabilistycznych*, na których się skupimy.

Jako referencyjny test pierwszości wykorzystamy prosty algorytm sprawdzający po kolei, czy liczby naturalne z zakresu $2,...,\lfloor\sqrt{n}\rfloor$ są dzielnikami $n$.

In [2]:
def prime_naive(n):
    if n in [2, 3, 5]:
        return True
    if n < 2 or n % 2 == 0 or n % 3 == 0:
        return False

    i = 5
    while(i * i <= n):
        if n % i == 0:
            return False
        if n % (i+2) == 0:
            return False
        i += 6
    return True

def eratostenes(n):
    is_prime = [True for _ in range(n+1)]
    is_prime[1], is_prime[0] = False, False

    for i in range(2*2, n+1, 2):
        is_prime[i] = False
    
    i = 3
    while i * i <= n:
        if is_prime[i]:
            for j in range(i * i, n+1, i):
                is_prime[j] = False
        i += 2

    return is_prime
    
    

By przyspieszyć nieco obliczenia w pozostałych ćwiczeniach, możemy zapamiętać wyniki dla pewnego początkowego zakresu liczb.

In [3]:
SIZE = 1000000
IS_PRIME = [prime_naive(n) for n in range(SIZE+1)]
PRIMES = [n for n in range(SIZE+1) if IS_PRIME[n]]


In [4]:
erat = eratostenes(SIZE)

for i in range(SIZE):
    assert erat[i] == IS_PRIME[i]

In [5]:
def prime_ref(n):
    """
    Referencyjny test pierwszości, z którym porównywać będziemy inne metody.
    """
    if n <= SIZE:
        return erat[n]
    else:
        return prime_naive(n)

### Test Fermata

Małe twierdzenie Fermata mówi, że jeśli $p$ jest liczbą pierwszą i $0 < a < p$, to $a^{p-1} \equiv 1 \pmod p$. Fakt ten stanowi podstawę *testu pierwszości Fermata*. W teście tym by sprawdzić, czy dane $n$ jest liczbą pierwszą, losujemy $0 < a < p$ i sprawdzamy, czy $a^{n-1} \equiv 1 \pmod n$. Jeśli nie, możemy z pewnością stwierdzić, że $n$ nie jest liczbą pierwszą. Jeśli tak, to $n$ może być liczbą pierwszą (ale nie musi).

Test Fermata nie daje nam pewności, że liczba, która go pomyślnie przeszła jest liczbą pierwszą. By zwiększyć wiarygodność wyniku pozytywnego, możemy wykonać więcej niż jedną iterację, tzn. wylosować kilka wartości $a$. Jeśli test nie przejdzie dla którejkolwiek z nich, $n$ nie jest liczbą pierwszą.

#### Zadanie 1

1. Zaimplementuj test Fermata. Do obliczania $a^k \pmod N$ należy wykorzystać wbudowaną funkcję `pow`, np. `pow(a, k, mod=N)`
2. Przeanalizuj wyniki testu Fermata z jedną iteracją dla wszystkich liczb naturalnych z zakresu $\{1,\ldots,10^6\}$. Ile jest fałszywych wyników pozytywnych, tj. liczb złożonych, dla których test się powiódł? Czy sa jakieś fałszywe wyniki negatywne, tj. liczby pierwsze, dla których test się nie powiódł?
3. Ile iteracji potrzeba, by mieć rozsądne szanse na zero niepoprawnych wyników w teście z punktu 2?
4. Narysuj wykres przedstawiający ilość błędnie sklasyfikowanych liczb w teście z punktu 2 w zależności od liczby iteracji testu Fermata. Użyj skali logarytmicznej dla osi pionowej.

#### Rozwiązanie

In [15]:
import random
def fast_power_mod(a, k, p):
    if k == 0:
        return 1
    if k == 1:
        return a

    small_a = fast_power_mod(a, k//2, p)
    small_a = ((small_a % p) * (small_a % p)) % p
    if k % 2 == 1:
        small_a = (small_a * a) % p
    return small_a

def fast_power_mod_iter(a, k, p):
    result = 1
    a = a % p
    while k > 0:
        if k % 2 == 1:
            result = (result * a) % p
        a = (a * a) % p
        k //= 2
    return result


In [16]:
fermat_tested = [True for _ in range(SIZE+1)]
fermat_tested[0], fermat_tested[1] = False, False

for i in range(2, SIZE + 1):
    p = i
    x = random.randint(1, p-1)
    fermat_tested[i] = fast_power_mod(x, p-1, p) == 1

diff = [False for _ in range(SIZE+1)]
for i in range(SIZE+1):
    diff[i] = fermat_tested[i] ^ prime_ref(i)
print()

In [17]:
print(f"Number of errros: {len([x for x in diff if x])}/{SIZE}")
print(f"Number of false positives: {len([i for i in range(SIZE+1) if prime_ref(i) and not fermat_tested[i]])}/{SIZE}")

Number of errros: 442/1000000
Number of false positives: 0/1000000


### Test Millera-Rabina

W teście Millera-Rabina poza małym twierdzeniem Fermata wykorzystujemy również inną własność liczb pierwszych: jeśli $n$ jest liczbą pierwszą, to $\mathbb{Z}/n\mathbb{Z}$ jest ciałem, a więc wielomiany stopnia $q$ mają w nim co najwyżej $q$ pierwiastków. W szczególności, jeśli $n$ jest liczbą pierwszą $\neq 2$, równanie $x^2 \equiv 1 \pmod n$ ma dokładnie dwa rozwiązania: $1$ oraz $-1 \equiv n - 1 \pmod n$.

Niech $n - 1 = 2^s d$, $d$ nieparzyste. Z małego twierdzenia Fermata wiemy, że dla $0 < a < n$ zachodzi $a^{2^s d} = a^{n-1} \equiv 1 \pmod n$. Jako, że $a^{2^s d} = \left(a^{2^{s-1}d}\right)^2$, to $a^{2^{s-1}d}$ jest jednym z rozwiązań równania $x^2 \equiv 1 \pmod n$, a zatem są dwie możliwości:

- $a^{2^{s-1}d} \equiv -1 \pmod n$, albo
- $a^{2^{s-1}d} \equiv 1 \pmod n$

Aplikując powyższe rozumowanie do przypadku $a^{2^{s-1}d} \equiv 1 \pmod n$, widzimy że $a^{2^{s-2}d} \equiv \pm 1 \pmod n$ i tak dalej aż do $a^d \equiv \pm 1 \pmod n$. Stąd, albo $a^d \equiv 1 \pmod n$, albo któreś z $a^d, a^{2d}, a^{4d}, \ldots, a^{2^{s-1}d}$ jest równe $-1 \pmod n$.

#### Algorytm

Na podstawie tej własności liczby pierwszej możemy zbudować następujący algorytm testujący pierwszość:

1. znajdź $s$, $d$ takie, że $n - 1 = 2^s d$, $d$ nieparzyste
2. powtórz $k$ razy:
    - wylosuj $2 \leq a \leq n - 2$
    - $x \gets a^d \pmod n$
    - jeśli $x \equiv 1 \pmod n$, zakończ iterację (2)
    - powtórz $s$ razy:
        - jeśli $x \equiv -1 \pmod n$, zakończ iterację (2)
        - $x \gets x^2 \pmod n$
    - zwróć "liczba złożona"
3. żaden z testów nie wykrył złożoności, zwróć "liczba pierwsza"

#### Zadanie 2

1. Zaimplementuj test Millera-Rabina.
2. Przeprowadź dla testu Millera-Rabina analizę analogiczną do tej z Zadania 1 (punkty 2-4). Porównaj wyniki obu metod.

#### Rozwiązanie

In [ ]:
def factor_twos(n):
    m = n
    two_count = 0
    while m % 2 == 0:
        m //= 2
        two_count += 1
    return m, two_count

def rabin_miller(n, k):
    if n <= 1:
        return False
    if n in [2, 3]:
        return True
    
    d, s = factor_twos(n-1)
    for _ in range(k):
        a = random.randint(2, n-2)
        if d < 100:
            x = fast_power_mod(a, d, n)
        else:
            x = fast_power_mod_iter(a, d, n)
            
        if x == 1 or x == n-1:
            continue
        for _ in range(s-1):
            x = (x * x) % n
            if x == n-1:
                break
        else:
            return False # if the for loop came to an end without break, that means number didn't pass the test
    return True


rabin_tested = [True for _ in range(SIZE+1)]
rabin_tested[0], rabin_tested[1] = False, False
diff_rm = [False for _ in range(SIZE + 1)]

for i in range(SIZE + 1):
    rabin_tested[i] = rabin_miller(i, 5)

for i in range(SIZE+1):
    if i % 10000 == 0:
        print(i, end = ' ')
    diff_rm[i] = rabin_tested[i] ^ prime_ref(i)

0 10000 20000 30000 40000 50000 60000 70000 80000 90000 100000 110000 120000 130000 140000 150000 160000 170000 180000 190000 200000 210000 220000 230000 240000 250000 260000 270000 280000 290000 300000 310000 320000 330000 340000 350000 360000 370000 380000 390000 400000 410000 420000 430000 440000 450000 460000 470000 480000 490000 500000 510000 520000 530000 540000 550000 560000 570000 580000 590000 600000 610000 620000 630000 640000 650000 660000 670000 680000 690000 700000 710000 720000 730000 740000 750000 760000 770000 780000 790000 800000 810000 820000 830000 840000 850000 860000 870000 880000 890000 900000 910000 920000 930000 940000 950000 960000 970000 980000 990000 1000000 

In [19]:
print(f"Number of errros: {len([x for x in diff_rm if x])}/{SIZE}")
print(f"Number of false positives: {len([i for i in range(SIZE+1) if prime_ref(i) and not fermat_tested[i]])}/{SIZE}")

Number of errros: 0/1000000
Number of false positives: 0/1000000


## Generowanie liczb pierwszych

### Duże liczby pierwsze dla kryptografii

W swojej książce "Applied Cryptography" Bruce Schneier proponuje następujący algorytm generowania $n$-bitowych liczb pierwszych:

1. Wygeneruj losową liczbę $n$-bitową
2. Ustaw pierwszy i ostatni bit na $1$ (by zapewnić że jest duża i nieparzysta)
3. Sprawdź, czy wygenerowana liczba jest podzielna przez małe liczby pierwsze, np. wszystkie mniejsze niż $2000$
4. Przeprowadź dla wygenerowanej liczby test Millera-Rabina z $5$ iteracjami
5. Jeśli wygenerowana liczba nie przeszła któregoś z testów (3) i (4), wróć do (1)

#### Zadanie 3

1. Zaimplementuj opisany powyżej schemat generacji liczb pierwszych. Do generacji losowej liczby $k$-bitowej można wykorzystać funkcję `getrandbits` z modułu `random`.
2. Wygeneruj przy jego pomocy liczbę pierwszą o rozmiarze 2048 bitów

#### Rozwiązanie

In [20]:
def gen_n_bits(n):
    n_bits = random.getrandbits(n)
    n_bits |= 1
    n_bits |= (1 << (n - 1))
    return n_bits

def check_prime(num, thresh = 2000):
    for i in range(thresh):
        if prime_ref(i) and num % i == 0:
            return False
    return rabin_miller(num, 5)

def find_n_bits_prime(n):
    while True:
        num = gen_n_bits(n)
        print(f"checking...")
        print(num)
        if check_prime(num):
            print("=" * 50 + "FOUND" + "=" * 50)
            return num

print(find_n_bits_prime(2048))

checking...
31262751843582554186435414344736597650329474672409896571943809968947811902213220312125424064091416345436891025826056744817316895596450085427111027962030124336949837695825679651736267476934739433691036157224483023535740203292197765267871719760040355348072534199668957455663929378659923283637000393601752399975225096606162930474325358584515333236869901944414294659641331247355870522487732300367165077757788687719589344380917429112886983045683413716304027064330357864331613988866626275375572931693961809917904513957334615745455649868574813120124253674977949753790669311361434994250263378995806745469263827505998908633119
checking...
2321991798670090166832191879013358674930903164330310395492706124186551271083425049448800492833710118623306602158900260963919759214760087066251517017670264091739690401774389686115685756505186057684691688270158922641289305538949509517529293774726141225728269405231798848212557141748382138302690508589654411694181043939149844788185001212584164856982694073973460

### Jeszcze większe liczby pierwsze dla przyjemności

Największa znana obecnie ludzkości liczba pierwsza to $2^{136279841} - 1$, posiadająca ponad 40 milionów cyfr dziesiętnych. Podobnie jak większość największych znanych liczb pierwszych, jest ona **liczbą Mersenne'a**, tj. ma postać
$$
M_n = 2^n - 1
$$
dla pewnego $n$. Nie wszystkie takie liczby są liczbami pierwszymi - jeśli $M_n$ jest liczbą pierwszą, to $n$ też musi być liczbą pierwszą, ale nie jest to warunek wystarczający - np. $M_{11} = 2047 = 23 \times 89$. Dla liczb Mersenne'a istnieją niemniej specjalne metody testowania pierwszości znacznie bardziej wydajne niż testy pierwszości ogólnego przeznaczenia. Najbardziej wydajną znaną metodą jest **test Lucasa-Lehmera**.

Test Lucasa-Lehmera działa następująco: dla zadanej nieparzystej liczby pierwszej $p$ konstruujemy rekurencyjnie ciąg $S_k$ tak, że

- $S_0 = 4$
- $S_k = S_{k-1}^2 - 2$

Wówczas $M_p = 2^p - 1$ jest liczbą pierwszą wtedy i tylko wtedy, gdy $S_{p-2} \equiv 0 \pmod {M_p}$

#### Zadanie 4

1. Zaimplementuj test Lucasa-Lehmera
2. Sprawdź, czy $M_3$, $M_5$, $M_{17}$, $M_{23}$, $M_{41}$, $M_{61}$, $M_{109}$, $M_{127}$, $M_{2459}$, $M_{3217}$ są liczbami pierwszymi
3. Porównaj czas działania testu Lucasa-Lehmera do czasu działania pojedynczej iteracji testu Millera-Rabina dla $M_{23209}$
4. (**dodatkowe**) Spróbuj wyliczyć jak największą liczbę pierwszą w czasie dostępnym na laboratorium. *Uwaga*: algorytm można przyspieszyć zamieniając dzielenie modulo $M_p$ na operacje binarne korzystając z tożsamości $k \equiv (k \mod 2^p) + \lfloor k / 2^p \rfloor$ - szczegóły [na Wikipedii](https://en.wikipedia.org/wiki/Lucas%E2%80%93Lehmer_primality_test)

#### Rozwiązanie

In [22]:
import time
TO_TEST = [3, 5, 17, 23, 41, 61, 109, 127, 2459, 3217]

def lucas_lehmer(p):
    if p == 2:
        return True
    M_p = (1 << p) - 1
    S = 4
    for _ in range(p - 2):
        S = (S * S - 2) % M_p
    return S == 0

for t in TO_TEST:
    print(f"{t} : {lucas_lehmer(t)}")

large_num = 23209
time1 = time.time()
print("\n")
print(f"Rabin-Miller result: {rabin_miller((1 << large_num) - 1, 1)} , time: {round(time.time() - time1, 2)}")
time1 = time.time()
print(f"Lucas-Lehmer result: {lucas_lehmer(large_num)} , time: {round(time.time() - time1, 2)}")


3 : True
5 : True
17 : True
23 : False
41 : False
61 : True
109 : False
127 : True
2459 : False
3217 : True


Rabin-Miller result: True , time: 33.73
Lucas-Lehmer result: True , time: 16.15


## Elementy odwrotne w $(\mathbb{Z}/n\mathbb{Z})^\times$

Standardowy algorytm Euklidesa pozwala nam obliczyć największy wspólny dzielnik liczb $a$, $b$. Jego działanie sprowadza się do obliczenia ciągu reszt z dzielenia:
$$
\begin{aligned}
r_0 &= a, r_1 = b \\
r_2 &= r_0 \mod r_1\\
r_3 &= r_1 \mod r_2\\
\ldots & \\
r_n &= r_{n-2} \mod r_{n-1} \\
\ldots &
\end{aligned}
$$
aż do momentu, gdy $r_n = 0$, wówczas $r_{n-1}$ to szukana wartość $d = (a, b)$. Z lematu Bézouta wiemy, że istnieją liczby całkowite $s$, $t$ takie, że $sa + tb = 1$. Rozszerzony algorytm Euklidesa pozwala nam je wyliczyć. Poza ciągiem $\{r_k\}$ obliczamy w nim również ciągi pomocnicze $\{s_k\}$, $\{t_k\}$ o własności
$$
s_k a + t_k b = r_k
$$
W każdej iteracji algorytmu mając dane wartości $s_{k-1}, t_{k-1}, r_{k-1}$ oraz $s_k, t_k, r_k$, możemy obliczyć $s_{k+1}, t_{k+1}, r_{k+1}$ w następupjący sposób. Wiemy, że $r_{k-1} = q r_k + r_{k+1}$ dla pewnego $q$, konkretnie $q = \lfloor r_{k - 1} / r_k\rfloor$. Stąd, $r_{k+1} = r_{k-1} - q r_k$. Skoro zaś
$$
r_{k-1} = s_{k-1} a + t_{k-1} b, \quad r_k = s_k a + t_k b
$$
to również
$$
\begin{aligned}
r_{k+1} &= s_{k-1} a + t_{k-1} b - q(s_k a + t_k b) \\&=
(s_{k-1} - q s_k)a + (t_{k-1} - q t_k) b
\end{aligned}
$$
a zatem możemy przyjąć $s_{k+1} = s_{k-1} - q s_k$ oraz $t_{k+1} = t_{k-1} - q t_k$. Ostatecznie mamy zatem
$$
\begin{aligned}
r_{k+1} &= r_{k-1} - q r_k \\
s_{k+1} &= s_{k-1} - q s_k \\
t_{k+1} &= t_{k-1} - q t_k \\
\end{aligned}
$$

Jeśli $(a, n) = 1$, to mając wyliczone rozszerzonym algorytmem Euklidesa liczby całkowite $s$, $t$ takie, że $sa + tn = 1$, możemy wyliczyć element odwrotny do $a$ w $\mathbb{Z} / n\mathbb{Z}$ jako $a^{-1} \equiv s \pmod n$, jako że $n$ dzieli $sa - 1 = tn$.

#### Zadanie 5

1. Zaimplementuj rozszerzony algorytm Euklidesa wyznaczający dla zadanych $a, b \in \mathbb{Z}$ wartość $(a, b)$ oraz $s, t \in \mathbb{Z}$ takie, że $sa + tb = (a, b)$.
2. Zaimplementuj funkcję znajdującą dla danego $a$ element odwrotny modulo $n$
3. Przetestuj działanie powyższej funkcji dla kilku dużych (np. $1024$ bitowych) modułów.

#### Rozwiązanie

In [32]:
def extended_gcd(a, b):
    r1, r2 = a, b
    s1, s2 = 1, 0
    t1, t2 = 0, 1

    while r2 != 0:
        q = r1//r2
        r1, r2 = r2, r1 % r2
        s1, s2 = s2, s1 - q * s2
        t1, t2 = t2, t1 - q * t2

    return r1, s1, t1 # because r2 = 0 and s2, t2 are just helper coeffs

def inverse_mod(a, n):
    g, s, t = extended_gcd(a, n)
    if g != 1:
        raise ValueError("gcd not 1")
    return s % n

g, s, t = extended_gcd(240, 46)
# print(g, s, t)
# print(s*240 + t*46)   


big_num = random.getrandbits(1024) | 1
big_num |= (1 << 1023)
aa = random.randint(2, big_num-1)

inv_aa = inverse_mod(aa, big_num)
assert (inv_aa * aa) % big_num == 1
print(aa)
print(inv_aa)



63883827288592808952466874422363001327313894198426155865137880010336723863530916119852212357945654812990246856728944954237791617997260498160050975714117362164396826336101697706329237337722975907350374001446342074467886725788198150422611203460871256328262276061220798157739759729853702373878991606310230440335
24643850146545515977081471463955703131245612723848458119583910713189084317747505493717795701971982423743503752127545548766250431807120898830191239311107605310899581342087624038077495280851137409832353916724223904503211927432522819243789050796618024254990110257977201815993885734585838477341618327439504649584
